# discriminator-classifier-head — ex1: flatten + linear + sigmoid scalar real/fake head

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `discriminator-classifier-head`. Running the final beacon cell reports progress against the `GAN: Discriminator classifier head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Discriminator classifier head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`discriminator-classifier-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "discriminator-classifier-head"
DD_SUBTOPIC = "GAN: Discriminator classifier head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Discriminator classifier head — quick refresher

The last layer of a DCGAN discriminator turns a `(B, C, H, W)` feature map into a scalar probability per sample — the answer to 'is this real?'.

```python
x = features.flatten(start_dim=1)   # (B, C*H*W)
logits = self.classifier(x)         # (B, 1)
probs = t.sigmoid(logits).squeeze(-1)  # (B,)
```

**Sigmoid, not softmax.** Real-vs-fake is BINARY — there is exactly one positive class. Sigmoid produces `P(real)` directly; `P(fake) = 1 - P(real)`. Softmax over a single output is degenerate.

**Numerical note.** In practice you'd return the LOGITS and pair them with `F.binary_cross_entropy_with_logits` for the loss — that fuses the sigmoid with the BCE for numerical stability. The explicit `sigmoid` here is for INFERENCE / inspection.

**Flatten before the Linear.** A `(B, 1024, 4, 4)` feature map has `B * 16384` numbers. The classifier is `nn.Linear(16384, 1)` — expects a 2-D input. `.flatten(start_dim=1)` collapses the channel and spatial axes into one feature axis without touching the batch.

### Exercise 1 — flatten + linear + sigmoid scalar real/fake head

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `flatten(start_dim=1) -> Linear(features, 1) -> sigmoid -> squeeze(-1)` to map a `(B, C, H, W)` feature map to a `(B,)` real-vs-fake probability vector.
> Keywords: gan, discriminator, sigmoid, flatten, head
> ```

**KCs targeted:** `flatten-from-axis-1`, `sigmoid-binary-prob`

Implement `ex1_discriminator_head(features, weight, bias)`. The scalar real/fake probability head:

1. `features` has shape `(B, C, H, W)` — output of the last discriminator block.
2. `weight` has shape `(1, C * H * W)` and `bias` has shape `(1,)` — together they parameterize `nn.Linear(C * H * W, 1)`.
3. Flatten everything except the batch axis: `flat = features.flatten(start_dim=1)` → `(B, C * H * W)`.
4. Affine to scalar: `logits = flat @ weight.T + bias` → `(B, 1)`.
5. Sigmoid then squeeze the trailing axis: `probs = t.sigmoid(logits).squeeze(-1)` → `(B,)`.
6. Return the `(B,)` probability vector — all values in `(0, 1)`.

Input: `features` `(B, C, H, W)`; `weight` `(1, C*H*W)`; `bias` `(1,)`.
Output: `(B,)` float tensor with values in `(0, 1)`.

The visualization plots the histogram of probabilities the head produces on a synthetic mixture of 'real-leaning' and 'fake-leaning' feature maps.

In [ ]:
def ex1_discriminator_head(features: Tensor, weight: Tensor, bias: Tensor) -> Tensor:
    flat = features.flatten(start_dim=1)             # (B, C*H*W)
    logits = flat @ weight.T + bias                  # (B, 1)
    return t.sigmoid(logits).squeeze(-1)             # (B,)


<details><summary>Solution</summary>

```python
def ex1_discriminator_head(features: Tensor, weight: Tensor, bias: Tensor) -> Tensor:
    flat = features.flatten(start_dim=1)             # (B, C*H*W)
    logits = flat @ weight.T + bias                  # (B, 1)
    return t.sigmoid(logits).squeeze(-1)             # (B,)
```

**Why `flatten(start_dim=1)`.** Collapses the channel + spatial axes into one feature axis while preserving the batch axis. `features.view(B, -1)` does the same thing more cryptically. Either works on contiguous tensors.

**Sigmoid, not softmax.** The output is binary (real vs fake) with ONE positive class. `P(fake) = 1 - P(real)` is implicit. Softmax over a single output is degenerate (always 1).

**Returning probs vs logits.** In a real training loop you'd return LOGITS and pair with `F.binary_cross_entropy_with_logits` for numerical stability (fuses sigmoid with BCE, avoids `log(0)` when sigmoid saturates). The explicit `sigmoid` here is for INFERENCE — to interpret the output as a probability — and to match the per-step inspection ARENA uses while debugging.

**`.squeeze(-1)` matters.** Without it the output is `(B, 1)`, which broadcasts against `(B,)` labels in confusing ways. The squeeze pins the head to `(B,)` so downstream code is unambiguous.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()